# Results from the pipeline

This notebook displays the final result of the pipeline, for a given sequence of plates.

In [ ]:
import os
import json
from importlib import reload
import warnings

import numpy as np

from astropy.table import Table
from astropy.utils.exceptions import AstropyWarning

import settings
from settings import get_parameters, current_dataset, fname, current_sequence
from library import update_dataset, plot_analysis_results, exceeds_criteria

In [ ]:
warnings.simplefilter('ignore', category=AstropyWarning)
warnings.simplefilter('ignore', category=UserWarning)

In [ ]:
# input sequence
seq_key = current_sequence
print("Sequence: ", seq_key)

In [ ]:
results_name = fname('pipeline_final_' + seq_key + '.fits')

table_results = Table.read(results_name, format='fits')

In [ ]:
table_results

In [ ]:
# radii where to compute profiles
edge_radii = np.arange(25) / 2.

# range in flux to pick up profile comparison stars 
flux_range = 0.1

## Explanation of figures

The two top plots depict the first and second plates at the position of a vanished object (the *target*).

The inset displays the central object with a logarithmic scale between min and max of the data. This emphasizes structures in the central peak. 

The mid left plot depicts the neighborhood around the target that was used to select comparison stars within a small range of brightness that centers at the target's brightness. This is necessary because profiles depend on brightness. Some of the stars in this plot may not be properly fitted (by a Gaussian), and so are removed from the subsequent profile and stats calculations and report.

The mid right plot depicts the radial profiles of both the target, and the neighboring stars that were properly fitted, as well as some stats. The FWHM-related quantities come from Gaussian fitting, **Elongation** comes from sextractor (provided in the APPLAUSE source tables), and the RMS profile difference is compute by notebook *psf_analysis.ipynb*. It measures a root-mean-square difference between the target's profile, and the average profile of the fitted stars in the neighborhood. The remaining values are derived from shape analysis as performed by the OpenCV library:

 - **Circularity** is derived from the area/perimeter ratio of the 30% isophote (1.0 is a perfect circle); 
 - **Area** is the pixel area inside that isophote; 
 - **Shape defect** is the sum of all deviations of that isophote from the smallest enclosing convex shape (0. signals perfect convexity);
 - **RMS circle dev** is the RMS deviation of that isophote from the smallest enclosing circle. It is expressed in relative units to the radius of that circle. 
    
Asterisks denote an ill-determined FWHM.

Shape-related quantities are less reliable for areas below ~100 px.

Gaussian fitting is used throughout, although one can argue that, if the actual PSF produced by the telescope optics and atmosphere, is a Gaussian, a star image in a photographic plate would be better represented by a quadratic. As a consequence, Gaussian fitting will tend to underestimate FWHM. That shouldn't be a big problem here though, since we handle both the target's and the star's images in the same way. 

Only objects that have the RMS profile distance smaller than a threshold, and have the circularity larger than a threshold, were selected. The theshold values are defined in the *parameters* dict in file *settings.py*.

The bottom sequence of thumbnail plots displays the central peak of the comparison stars, and the target object, with a logarithmic scale. The number at the top contains the last digits of the source ID; the numbers at the bottom are the annular bin where the star is on the plate, and the plate ID (redundant in this display).

In [ ]:
for row_index in range(len(table_results)):
    
    # to plot profiles, we need the original table for the current dataset
    plate_id_str = str(table_results['plate_id_1'][row_index]) 
    next_plate_id_str = str(table_results['next_plate_id'][row_index])
    
    key = plate_id_str + ',' + next_plate_id_str

    update_dataset(key)
    reload(settings)
    from settings import current_dataset
    par = get_parameters(current_dataset)

    table_matched = Table.read(fname(par['table_matched']), format='fits')

    # skip row if criteria are exceeded - strictly not needed, but adds flexibility when testing
    if exceeds_criteria(table_results, row_index, par):
        continue

    plot_analysis_results(table_results, table_matched, row_index, par, flux_range, edge_radii)
